# Principal Components Analysis Nutrition Exercise
En este ejercicio no vas a montar ningún modelo de Machine Learning supervisado, sino aprenderás a trabajar con PCA: pretratar el dato antes de calcular sus PCs, interpretarlos, graficar y escoger número de componentes según varianza.

Importa las librerías necesarias

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

### Importa los datos
1. Importa los datos: *nndb.csv*
2. Observa las columnas que hay, así como su tipo.

In [7]:
df = pd.read_csv('data/nndb_flat.csv')

In [8]:
df.shape

(8618, 45)

In [9]:

df.head()

,ID,FoodGroup,ShortDescrip,Descrip,CommonName,MfgName,ScientificName,Energy_kcal,Protein_g,Fat_g,...,Folate_USRDA,Niacin_USRDA,Riboflavin_USRDA,Thiamin_USRDA,Calcium_USRDA,Copper_USRDA,Magnesium_USRDA,Phosphorus_USRDA,Selenium_USRDA,Zinc_USRDA
0,1001,Dairy and Egg Products,"BUTTER,WITH SALT","Butter, salted",NaN,NaN,NaN,717.0,0.85,81.11,...,0.0075,0.002625,0.026154,0.004167,0.020000,0.000000,0.004762,0.034286,0.018182,0.008182
1,1002,Dairy and Egg Products,"BUTTER,WHIPPED,WITH SALT","Butter, whipped, with salt",NaN,NaN,NaN,717.0,0.85,81.11,...,0.0075,0.002625,0.026154,0.004167,0.020000,0.000018,0.004762,0.032857,0.018182,0.004545
2,1003,Dairy and Egg Products,"BUTTER OIL,ANHYDROUS","Butter oil, anhydrous",NaN,NaN,NaN,876.0,0.28,99.48,...,0.0000,0.000188,0.003846,0.000833,0.003333,0.000001,0.000000,0.004286,0.000000,0.000909
3,1004,Dairy and Egg Products,"CHEESE,BLUE","Cheese, blue",NaN,NaN,NaN,353.0,21.40,28.74,...,0.0900,0.063500,0.293846,0.024167,0.440000,0.000044,0.054762,0.552857,0.263636,0.241818
4,1005,Dairy and Egg Products,"CHEESE,BRICK","Cheese, brick",NaN,NaN,NaN,371.0,23.24,29.68,...,0.0500,0.007375,0.270000,0.011667,0.561667,0.000027,0.057143,0.644286,0.263636,0.236364


### Mira a ver la correlación entre las variables numéricas
¿Qué pasa con las columnas USRDA? ¿Hay que tomar alguna decisión?

In [11]:
# Guardar la columna FoodGroup
food_groups = df['FoodGroup']

# Eliminar columnas de texto y el ID
text_cols = ['FoodGroup', 'ShortDescrip', 'Descrip', 'CommonName', 'MfgName', 'ScientificName', 'ID']
df_numeric = df_numeric.drop(columns=text_cols, errors='ignore')

print("Columnas numéricas restantes:", df_numeric.columns.tolist())

NameError: name 'df_numeric' is not defined

### Variables no numéricas
Elimina las variables de texto del dataset

### Distribuciones
Muchas de las variables tienen asimetría hacia la derecha. Deberíamos transformarlas para conseguir distribuciones normales y mejorar las correlaciones de cara al PCA y a futuros modelos lineales que vayamos a probar. Transforma todas las variables realizando una transformación Logarítmica.

In [12]:
# Aplicar transformación Logarítmica (log1p para manejar ceros)
df_log = np.log1p(df_numeric)

# Visualización del efecto (ejemplo con Azúcar)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df_numeric['Sugar_g'], kde=True)
plt.title('Distribución Original (Sugar_g)')
plt.subplot(1, 2, 2)
sns.histplot(df_log['Sugar_g'], kde=True)
plt.title('Transformación Log (Sugar_g)')
plt.show()

NameError: name 'df_numeric' is not defined

### Estandarizado
Estandariza cada variable.

No es necesario que dividas en train y test.

In [13]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_log)

# Convertimos a DataFrame para mantener nombres de columnas
X_scaled_df = pd.DataFrame(X_scaled, columns=df_numeric.columns)

NameError: name 'df_log' is not defined

### Implementación del PCA
Prueba a montar un PCA con todos los componentes. Para ello utiliza los datos previamente transformados y escalados.

In [14]:
pca = PCA()
pca.fit(X_scaled_df)

NameError: name 'X_scaled_df' is not defined

### Aportación de cada PCA
Visualiza en un diagrama de líneas la suma acumulada de la varianza explicativa del PCA.

Si tuviéses que quedarte con 70-75 % de la varianza original, ¿con cuántos Principal Components te quedarías?

In [15]:
# Varianza acumulada
explained_variance = np.cumsum(pca.explained_variance_ratio_)

# Gráfico
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o', linestyle='--')
plt.axhline(y=0.75, color='r', linestyle='-', label='75% Varianza Explicada')
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Explicada Acumulada')
plt.legend()
plt.grid(True)
plt.show()

# Cálculo exacto
n_components_75 = np.argmax(explained_variance >= 0.75) + 1
print(f"Número de componentes para >= 75% varianza: {n_components_75}")

AttributeError: 'PCA' object has no attribute 'explained_variance_ratio_'

## Feat. Red
Crea un nuevo PCA con X Principal Components, siendo X la cantidad de PC escogidos en el apartado anterior.

Obtén el nuevo dataset con el mismo número de registros que el original, pero en este caso con X features, que representan los PC elegidos.

In [18]:
# PCA con los componentes seleccionados
pca_red = PCA(n_components=n_components_75)
X_pca = pca_red.fit_transform(X_scaled_df)

# Crear DataFrame con los resultados
pc_columns = [f'PC{i+1}' for i in range(n_components_75)]
df_pca = pd.DataFrame(X_pca, columns=pc_columns)

NameError: name 'n_components_75' is not defined

### ¿Qué grupo de comida tiene los valores más altos en cada categoría?
Determina para cada Principal Component, los 3 grupos de comida (*FoodGroup*) con los valores del PC más altos.

In [17]:
# Unir con FoodGroup
df_pca_final = pd.concat([df_pca, food_groups], axis=1)

# Mostrar los 3 grupos con valores más altos para cada PC
for col in pc_columns:
    print(f"--- Top 3 Grupos de comida para {col} ---")
    print(df_pca_final.groupby('FoodGroup')[col].mean().sort_values(ascending=False).head(3))
    print("\n")

NameError: name 'df_pca' is not defined